In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import umap.umap_ as umap
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from pathlib import Path

RANDOM_STATE = 231026

/Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
data = pd.read_csv("all_sources_isc_vgg_embeddings.csv")


In [8]:
data

,source,label,Value,vgg_0,vgg_1,vgg_2,vgg_3,vgg_4,vgg_5,vgg_6,...,vgg_502,vgg_503,vgg_504,vgg_505,vgg_506,vgg_507,vgg_508,vgg_509,vgg_510,vgg_511
0,mtt,C,94.774444,0.011359,0.241203,0.344854,0.000000,11.211005,0.004637,0.0,...,0.188881,0.0,0.954637,2.482550,0.000000,0.118003,0.000000,0.000000,1.703285,0.000000
1,mtt,C,94.774444,0.025853,0.000000,1.069848,0.000000,11.320415,0.000000,0.0,...,0.000000,0.0,0.168406,6.162180,0.000000,0.107752,0.000000,0.271620,4.474264,0.000000
2,mtt,C,94.774444,0.480963,0.000000,3.107252,0.000000,4.831994,0.064862,0.0,...,0.124030,0.0,0.665731,1.306546,0.030879,0.144269,0.005907,0.232978,1.154307,0.000000
3,mtt,C,94.774444,0.000000,0.000000,0.058814,0.000000,6.624664,0.000000,0.0,...,0.316281,0.0,0.318968,0.330109,0.480029,0.542135,0.000000,0.008872,3.581180,0.289227
4,mtt,C,94.774444,0.000000,0.000000,0.119758,0.000000,8.961069,0.000000,0.0,...,0.099849,0.0,0.034700,1.626213,0.012560,0.605835,0.000000,0.025230,1.533522,0.180085
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23840,study18,S,NaN,0.059749,0.000000,0.000000,0.312913,0.053099,0.000000,0.0,...,0.198128,0.0,0.024524,0.771849,0.000000,0.002295,0.000000,0.000000,0.800087,0.000000
23841,study18,S,NaN,0.000000,0.000000,1.000184,0.000000,0.740112,0.000000,0.0,...,0.434452,0.0,0.000000,0.732429,0.000000,0.662563,0.000000,0.000000,0.678243,0.000000
23842,study18,S,NaN,0.000000,0.000000,0.677755,0.000000,2.872867,0.000000,0.0,...,2.191196,0.0,0.023503,0.882949,0.000000,0.059748,0.000000,0.000000,0.889415,0.000000
23843,study18,S,NaN,0.000000,0.000000,0.020610,0.000000,2.368106,0.035866,0.0,...,0.834328,0.0,0.051658,0.039791,4.803977,0.000000,0.000000,0.000000,1.185690,0.000000


In [6]:
data.describe()

,Value,vgg_0,vgg_1,vgg_2,vgg_3,vgg_4,vgg_5,vgg_6,vgg_7,vgg_8,...,vgg_502,vgg_503,vgg_504,vgg_505,vgg_506,vgg_507,vgg_508,vgg_509,vgg_510,vgg_511
count,982.000000,23845.000000,23845.000000,23845.000000,23845.000000,23845.000000,23845.000000,23845.000000,23845.000000,23845.000000,...,23845.000000,23845.000000,23845.000000,23845.000000,23845.000000,23845.000000,23845.000000,23845.000000,23845.000000,23845.000000
mean,88.145686,1.498470,0.304781,1.085484,0.355538,0.439499,0.131011,0.114659,0.667286,0.026225,...,1.339190,0.166183,1.608867,0.542775,2.118624,0.280964,0.391336,0.088719,1.003547,0.173210
std,61.962304,2.484030,0.705270,1.437526,0.591818,0.949694,0.288222,0.292859,1.165890,0.125531,...,1.733397,0.278504,2.038703,0.907855,1.650250,0.588996,0.685252,0.279512,0.648554,0.408416
min,2.454000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,23.575000,0.000000,0.000000,0.072552,0.000000,0.000000,0.000000,0.000000,0.005921,0.000000,...,0.307238,0.000000,0.171718,0.029046,0.930009,0.000000,0.000000,0.000000,0.557403,0.000000
50%,94.945000,0.377835,0.000000,0.539538,0.098405,0.091179,0.000811,0.000000,0.172500,0.000000,...,0.838379,0.040099,0.808060,0.189393,1.762812,0.066034,0.064119,0.000000,0.863602,0.015121
75%,120.987500,1.899906,0.273758,1.556678,0.449837,0.510618,0.131086,0.089827,0.762958,0.000000,...,1.746428,0.220630,2.259020,0.676995,2.912449,0.317651,0.496297,0.017148,1.279614,0.148648
max,271.664444,26.383339,11.761838,15.077735,8.261716,16.409120,5.181079,5.634606,12.246122,4.030687,...,18.085924,4.166559,16.153600,16.235281,18.388905,8.562617,8.496418,5.308903,8.755614,8.346353


In [9]:
data.isna().sum()

source         0
label          0
Value      22863
vgg_0          0
vgg_1          0
           ...  
vgg_507        0
vgg_508        0
vgg_509        0
vgg_510        0
vgg_511        0
Length: 515, dtype: int64